# CXR-LT 2023

This notebook is organized around a few questions that make the dataset easier to reason about before modeling:

- How large is each split, and is there any subject/study leakage?
- Which labels are common, which are truly rare, and how long-tailed is the problem?
- How many findings does each image usually carry?
- Which view positions dominate the dataset, and how much metadata is missing?
- Do the sample submission files match the training label schema?


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

cwd = Path.cwd()
if cwd.name != "00-examine-data":
    raise Exception("Please run this notebook from the 00-examine-data directory")

root_dir = cwd.parents[1]
data_dir = root_dir / "data"
cxr_lt_2023_dir = data_dir / "CXR-LT" / "cxr-lt-multi-label-long-tailed-classification-on-chest-x-rays-2.0.0" / "cxr-lt-2023"

print(root_dir)
print(cxr_lt_2023_dir)


In [ ]:
CSV_FILES = {
    "train": "train.csv",
    "development": "development.csv",
    "test": "test.csv",
    "development_sample_submission": "development_sample_submission.csv",
    "test_sample_submission": "test_sample_submission.csv",
}

ID_COLUMNS = [
    "dicom_id",
    "subject_id",
    "study_id",
    "ViewPosition",
    "ViewCodeSequence_CodeMeaning",
    "path",
]


def load_dataset(filename: str) -> pd.DataFrame:
    return pd.read_csv(cxr_lt_2023_dir / filename)


datasets = {name: load_dataset(filename) for name, filename in CSV_FILES.items()}

train_df = datasets["train"]
dev_df = datasets["development"]
test_df = datasets["test"]
dev_sample_sub_df = datasets["development_sample_submission"]
test_sample_sub_df = datasets["test_sample_submission"]

analysis_splits = {
    "train": train_df,
    "development": dev_df,
    "test": test_df,
}

submission_splits = {
    "development_sample_submission": dev_sample_sub_df,
    "test_sample_submission": test_sample_sub_df,
}

label_cols = [column for column in train_df.columns if column not in ID_COLUMNS]

print(f"Loaded {len(datasets)} files")
print(f"Detected {len(label_cols)} labels")
label_cols


## Reusable helpers

The goal here is to keep the notebook exploratory without repeating the same summary code for every split.


In [ ]:
def preview_dataset(name: str, df: pd.DataFrame, head_rows: int = 3) -> None:
    print(f"\n{name}")
    print(f"shape: {df.shape}")
    print(f"duplicate dicom_id rows: {df['dicom_id'].duplicated().sum()}")
    display(df.head(head_rows))

    missing = (
        df.isna().sum()
        .rename("missing_count")
        .loc[lambda series: series > 0]
        .sort_values(ascending=False)
    )
    if missing.empty:
        print("No missing values detected.")
    else:
        missing_df = missing.to_frame().assign(
            missing_pct=lambda frame: frame["missing_count"] / len(df) * 100
        )
        display(missing_df)


def build_split_overview(split_frames: dict[str, pd.DataFrame], label_columns: list[str]) -> pd.DataFrame:
    rows = []
    for split_name, df in split_frames.items():
        positive_labels_per_image = df[label_columns].sum(axis=1)
        rows.append(
            {
                "split": split_name,
                "rows": len(df),
                "unique_subjects": df["subject_id"].nunique(),
                "unique_studies": df["study_id"].nunique(),
                "unique_dicoms": df["dicom_id"].nunique(),
                "missing_ViewPosition": df["ViewPosition"].isna().sum(),
                "missing_ViewCodeMeaning": df["ViewCodeSequence_CodeMeaning"].isna().sum(),
                "avg_labels_per_image": positive_labels_per_image.mean(),
                "median_labels_per_image": positive_labels_per_image.median(),
                "no_finding_rate_pct": df["No Finding"].mean() * 100,
            }
        )

    return pd.DataFrame(rows).sort_values("rows", ascending=False).reset_index(drop=True)


def summarize_labels(df: pd.DataFrame, split_name: str, label_columns: list[str]) -> pd.DataFrame:
    summary = (
        df[label_columns]
        .mean()
        .mul(100)
        .rename("positive_rate_pct")
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={"index": "label"})
    )
    summary.insert(0, "split", split_name)
    summary["positive_count"] = summary["label"].map(df[label_columns].sum().to_dict()).astype(int)
    return summary[["split", "label", "positive_count", "positive_rate_pct"]]


def plot_label_prevalence(split_frames: dict[str, pd.DataFrame], label_columns: list[str], top_n: int = 15) -> None:
    label_summary = pd.concat(
        [summarize_labels(df, split_name, label_columns) for split_name, df in split_frames.items()],
        ignore_index=True,
    )

    label_order = (
        label_summary.query("split == 'train'")
        .sort_values("positive_rate_pct", ascending=False)
        .head(top_n)["label"]
    )

    plot_df = label_summary[label_summary["label"].isin(label_order)]

    plt.figure(figsize=(10, 7))
    sns.barplot(
        data=plot_df,
        x="positive_rate_pct",
        y="label",
        hue="split",
        order=label_order,
    )
    plt.title(f"Top {top_n} labels by prevalence in train")
    plt.xlabel("Positive rate (%)")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()


def plot_labels_per_image(split_frames: dict[str, pd.DataFrame], label_columns: list[str]) -> None:
    fig, axes = plt.subplots(1, len(split_frames), figsize=(15, 4), sharey=True)

    for ax, (split_name, df) in zip(axes, split_frames.items()):
        label_counts = df[label_columns].sum(axis=1).astype(int)
        counts = label_counts.value_counts().sort_index()
        ax.bar(counts.index, counts.values)
        ax.set_title(split_name)
        ax.set_xlabel("Positive labels per image")
        ax.set_ylabel("Image count")

    plt.tight_layout()
    plt.show()


def summarize_view_positions(df: pd.DataFrame, split_name: str, top_n: int = 8) -> pd.DataFrame:
    view_counts = (
        df["ViewPosition"]
        .fillna("Missing")
        .value_counts()
        .head(top_n)
        .rename_axis("ViewPosition")
        .reset_index(name="count")
    )
    view_counts.insert(0, "split", split_name)
    view_counts["rate_pct"] = view_counts["count"] / len(df) * 100
    return view_counts


def plot_view_positions(split_frames: dict[str, pd.DataFrame], top_n: int = 6) -> None:
    plot_df = pd.concat(
        [summarize_view_positions(df, split_name, top_n=top_n) for split_name, df in split_frames.items()],
        ignore_index=True,
    )

    plt.figure(figsize=(11, 5))
    sns.barplot(data=plot_df, x="ViewPosition", y="rate_pct", hue="split")
    plt.title(f"Top {top_n} view positions by split")
    plt.xlabel("")
    plt.ylabel("Rate (%)")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()


def compute_overlap(split_frames: dict[str, pd.DataFrame], key: str) -> pd.DataFrame:
    split_names = list(split_frames)
    key_sets = {split_name: set(df[key]) for split_name, df in split_frames.items()}

    overlap_matrix = pd.DataFrame(index=split_names, columns=split_names, dtype=int)
    for left_name in split_names:
        for right_name in split_names:
            overlap_matrix.loc[left_name, right_name] = len(key_sets[left_name] & key_sets[right_name])

    return overlap_matrix


def summarize_no_finding_conflicts(split_frames: dict[str, pd.DataFrame], label_columns: list[str]) -> pd.DataFrame:
    other_labels = [label for label in label_columns if label != "No Finding"]
    rows = []

    for split_name, df in split_frames.items():
        conflicts = ((df["No Finding"] == 1) & (df[other_labels].sum(axis=1) > 0)).sum()
        rows.append(
            {
                "split": split_name,
                "rows_with_no_finding": int(df["No Finding"].sum()),
                "rows_with_no_finding_and_other_label": int(conflicts),
                "conflict_rate_within_no_finding_pct": conflicts / max(df["No Finding"].sum(), 1) * 100,
            }
        )

    return pd.DataFrame(rows)


def summarize_submission_file(df: pd.DataFrame, split_name: str, label_columns: list[str]) -> pd.DataFrame:
    submission_label_columns = [column for column in df.columns if column != "dicom_id"]
    rows = [
        {
            "split": split_name,
            "rows": len(df),
            "label_columns_match_train": submission_label_columns == label_columns,
            "min_score": df[submission_label_columns].min().min(),
            "mean_score": df[submission_label_columns].mean().mean(),
            "max_score": df[submission_label_columns].max().max(),
        }
    ]
    return pd.DataFrame(rows)


## 1) Quick split overview

Start here to answer: how big are the splits, what fields exist, and where is metadata incomplete?


In [ ]:
for split_name, df in analysis_splits.items():
    preview_dataset(split_name, df)

split_overview_df = build_split_overview(analysis_splits, label_cols)
display(split_overview_df)


## 2) Label imbalance and multi-label density

This is the part that tells you why the task is called long-tailed. Pay attention to the rarest labels and to how many findings co-occur on each image.


In [ ]:
label_summary_df = pd.concat(
    [summarize_labels(df, split_name, label_cols) for split_name, df in analysis_splits.items()],
    ignore_index=True,
)

display(label_summary_df.query("split == 'train'").head(10))
display(label_summary_df.query("split == 'train'").tail(10))

plot_label_prevalence(analysis_splits, label_cols, top_n=23)
plot_labels_per_image(analysis_splits, label_cols)

display(summarize_no_finding_conflicts(analysis_splits, label_cols))


## 3) View metadata and split leakage checks

For modeling, it helps to know whether view-position distribution shifts across splits and whether the same patient or study appears in multiple splits.


In [ ]:
view_position_summary_df = pd.concat(
    [summarize_view_positions(df, split_name) for split_name, df in analysis_splits.items()],
    ignore_index=True,
)
display(view_position_summary_df)
plot_view_positions(analysis_splits, top_n=6)

for overlap_key in ["subject_id", "study_id", "dicom_id"]:
    print(f"\nOverlap for {overlap_key}")
    display(compute_overlap(analysis_splits, overlap_key))


## 4) Sample submission sanity checks

These files are handy for confirming the label order expected by the benchmark and for checking score ranges before writing an inference pipeline.


In [ ]:
submission_summary_df = pd.concat(
    [summarize_submission_file(df, split_name, label_cols) for split_name, df in submission_splits.items()],
    ignore_index=True,
)
display(submission_summary_df)

for split_name, df in submission_splits.items():
    print(f"\n{split_name}")
    display(df.head(3))


## What to inspect next

Once this notebook makes sense, the next useful passes are usually:

1. Study-level aggregation: group by `study_id` and check how often a study contains both PA/AP and lateral views.
2. Patient timeline behavior: sort by `subject_id`, `study_id` or acquisition time from MIMIC metadata and inspect repeat exams.
3. Label correlations: compute a co-occurrence matrix to see which findings often appear together.
4. View-conditioned prevalence: compare label frequencies for `AP`, `PA`, and `LATERAL` only.
5. Image availability and path sanity: verify how many `path` entries are actually present on disk before building dataloaders.
6. Text linkage readiness: join these rows against MIMIC report metadata so you can quantify how many studies have usable paired text.
